In [1]:
!wget https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip

--2025-10-05 13:42:35--  https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip
Resolving iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)... 140.109.20.133
Connecting to iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)|140.109.20.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2533317 (2.4M) [application/x-zip-compressed]
Saving to: ‘Revised_JNLPBA.zip’

Revised_JNLPBA.zip  100%[===================>]   2.42M  1.41MB/s    in 1.7s    

2025-10-05 13:42:38 (1.41 MB/s) - ‘Revised_JNLPBA.zip’ saved [2533317/2533317]



In [2]:
!unzip -n Revised_JNLPBA.zip -d Revised_JNLPBA

Archive:  Revised_JNLPBA.zip
  inflating: Revised_JNLPBA/Genia4EReval1.iob2  
  inflating: Revised_JNLPBA/Genia4EReval2.iob2  
  inflating: Revised_JNLPBA/Genia4ERtask1.iob2  
  inflating: Revised_JNLPBA/Genia4ERtask2.iob2  


In [3]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=274b7952b545abe852c09f732f78dcdda9734d982bffa4d4097941035f996cc0
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [4]:
from typing import List, Tuple
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from collections import Counter
from itertools import chain

In [5]:
import re

def read_iob(file_path):
    sentences, labels = [], []
    tokens, tags = [], []
    header_pattern = re.compile(r"^###MEDLINE:\d+")

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:  # new sentence
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
                continue

            if header_pattern.match(line):
                continue

            parts = line.split()
            if len(parts) == 2:
                word, tag = parts
                tokens.append(word)
                tags.append(tag)

        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels

train_sent1, train_labels1 = read_iob("Revised_JNLPBA/Genia4ERtask1.iob2")
train_sent2, train_labels2 = read_iob("Revised_JNLPBA/Genia4ERtask2.iob2")
eval_sent1, eval_labels1   = read_iob("Revised_JNLPBA/Genia4EReval1.iob2")
eval_sent2, eval_labels2   = read_iob("Revised_JNLPBA/Genia4EReval2.iob2")

train_sentences = train_sent1 + train_sent2
train_labels    = train_labels1 + train_labels2

test_sentences  = eval_sent1 + eval_sent2
test_labels     = eval_labels1 + eval_labels2

def print_example(tokens, tags, n=20):
    for t, l in zip(tokens[:n], tags[:n]):
        print(f"{t:20} {l}")

print_example(train_sentences[0], train_labels[0])

IL-2                 B-DNA
gene                 I-DNA
expression           O
and                  O
NF-kappa             B-protein
B                    I-protein
activation           O
through              O
CD28                 B-protein
requires             O
reactive             O
oxygen               O
production           O
by                   O
5-lipoxygenase       B-protein
.                    O


In [6]:
class NERDataset(Dataset):
    def __init__(self, examples: List[Tuple[List[str], List[str]]],
                 word_vocab, char_vocab, label_vocab, max_word_len=30):
        self.examples = examples
        self.wv = word_vocab
        self.cv = char_vocab
        self.lv = label_vocab
        self.max_word_len = max_word_len

    def __len__(self):
        return len(self.examples)

    def token_to_id(self, token):
        return self.wv.get(token, self.wv[UNK])

    def chars_to_ids(self, token):
        ids = [self.cv.get(ch, self.cv[UNK]) for ch in token[:self.max_word_len]]
        # pad
        ids = ids + [self.cv[PAD]] * (self.max_word_len - len(ids))
        return ids

    def __getitem__(self, idx):
        tokens, labels = self.examples[idx]
        w_ids = [self.token_to_id(t) for t in tokens]
        c_ids = [self.chars_to_ids(t) for t in tokens]  # seq_len x max_word_len
        l_ids = [self.lv[l] for l in labels]
        return torch.tensor(w_ids, dtype=torch.long), torch.tensor(c_ids, dtype=torch.long), torch.tensor(l_ids, dtype=torch.long), len(tokens)

def ner_collate(batch):
    # batch: list of tuples: (w_ids, c_ids, l_ids, length)
    lengths = [b[3] for b in batch]
    max_len = max(lengths)
    max_word_len = batch[0][1].size(1)
    ws, cs, ls, masks = [], [], [], []
    for w_ids, c_ids, l_ids, L in batch:
        pad_w = torch.full((max_len - L,), 0, dtype=torch.long)  # PAD index 0
        ws.append(torch.cat([w_ids, pad_w], dim=0))
        # c_ids: L x max_word_len
        pad_c = torch.full((max_len - L, max_word_len), 0, dtype=torch.long)
        cs.append(torch.cat([c_ids, pad_c], dim=0))
        pad_l = torch.full((max_len - L,), -100, dtype=torch.long)  # ignore_index for loss
        ls.append(torch.cat([l_ids, pad_l], dim=0))
        masks.append(torch.cat([torch.ones(L, dtype=torch.bool), torch.zeros(max_len-L, dtype=torch.bool)], dim=0))
    return torch.stack(ws), torch.stack(cs), torch.stack(ls), torch.stack(masks)

In [7]:
PAD = "<PAD>"
UNK = "<UNK>"

# word vocab
all_words = {w for s in train_sentences for w in s}
word_vocab = {w:i+2 for i,w in enumerate(all_words)}  # reserve 0=PAD, 1=UNK
word_vocab["<PAD>"] = 0
word_vocab["<UNK>"] = 1

# char vocab
all_chars = {ch for w in all_words for ch in w}
char_vocab = {c:i+2 for i,c in enumerate(all_chars)}
char_vocab["<PAD>"] = 0
char_vocab["<UNK>"] = 1

# label vocab
all_labels = {l for seq in train_labels for l in seq}
label_vocab = {l:i for i,l in enumerate(sorted(all_labels))}

train_examples = list(zip(train_sentences, train_labels))
test_examples  = list(zip(test_sentences, test_labels))

train_dataset = NERDataset(train_examples, word_vocab, char_vocab, label_vocab)
test_dataset  = NERDataset(test_examples, word_vocab, char_vocab, label_vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=ner_collate)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=ner_collate)

for ws, cs, ls, masks in train_loader:
    print("Word IDs:", ws.shape)   
    print("Char IDs:", cs.shape)   
    print("Label IDs:", ls.shape)  
    print("Masks:", masks.shape)   
    break

Word IDs: torch.Size([32, 51])
Char IDs: torch.Size([32, 51, 30])
Label IDs: torch.Size([32, 51])
Masks: torch.Size([32, 51])


In [8]:
class CharCNNWordBiLSTM(nn.Module):
    def __init__(self,
                 vocab_size,
                 char_vocab_size,
                 label_size,
                 word_emb_dim=200,
                 char_emb_dim=30,
                 char_out_channels=50,
                 char_kernel_size=3,
                 lstm_hidden=256,
                 lstm_layers=1,
                 dropout=0.3,
                 pretrained_word_emb=None):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, word_emb_dim, padding_idx=0)
        if pretrained_word_emb is not None:
            self.word_emb.weight.data[:pretrained_word_emb.shape[0]] = torch.tensor(pretrained_word_emb)

        self.char_emb = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_emb_dim, out_channels=char_out_channels, kernel_size=char_kernel_size, padding=1)
        self.dropout = nn.Dropout(dropout)
        input_dim = word_emb_dim + char_out_channels
        self.bilstm = nn.LSTM(input_dim, lstm_hidden//2, num_layers=lstm_layers, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(lstm_hidden, label_size)

    def forward(self, w_ids, c_ids, mask=None):
        bsz, seq_len = w_ids.size()
        word_emb = self.word_emb(w_ids)  
        b_s, s_l, max_w = c_ids.size()
        ch = c_ids.view(-1, max_w)  
        ch_emb = self.char_emb(ch)  
        ch_emb = ch_emb.transpose(1,2)  
        ch_conv = self.char_cnn(ch_emb)  
        ch_pool = torch.max(ch_conv, dim=2)[0]  
        ch_pool = ch_pool.view(b_s, s_l, -1)

        x = torch.cat([word_emb, ch_pool], dim=2)  
        x = self.dropout(x)
        lstm_out, _ = self.bilstm(x)  
        logits = self.classifier(self.dropout(lstm_out))
        return logits

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CharCNNWordBiLSTM(
    vocab_size=len(word_vocab),
    char_vocab_size=len(char_vocab),
    label_size=len(label_vocab),
    word_emb_dim=200,
    char_emb_dim=30,
    char_out_channels=50,
    lstm_hidden=256,
    dropout=0.3
).to(device)

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=-100) 
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for w_ids, c_ids, l_ids, mask in dataloader:
        w_ids = w_ids.to(device)
        c_ids = c_ids.to(device)
        l_ids = l_ids.to(device)
        optimizer.zero_grad()
        logits = model(w_ids, c_ids)
        # logits: b x seq_len x label_size
        b, s, lsize = logits.size()
        logits_flat = logits.view(-1, lsize)
        labels_flat = l_ids.view(-1)
        loss = criterion(logits_flat, labels_flat)  
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [12]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for ws, cs, ls, masks in train_loader:
        ws, cs, ls, masks = ws.to(device), cs.to(device), ls.to(device), masks.to(device)

        logits = model(ws, cs, mask=masks)   
        loss = criterion(logits.view(-1, logits.size(-1)), ls.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.2550
Epoch 2, Loss: 0.1022
Epoch 3, Loss: 0.0693
Epoch 4, Loss: 0.0511
Epoch 5, Loss: 0.0406
Epoch 6, Loss: 0.0327
Epoch 7, Loss: 0.0275
Epoch 8, Loss: 0.0236
Epoch 9, Loss: 0.0205
Epoch 10, Loss: 0.0186


In [13]:
from seqeval.metrics import classification_report

def evaluate_seqeval(model, dataloader, id2label, device):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for ws, cs, ls, masks in dataloader:
            ws, cs, ls, masks = ws.to(device), cs.to(device), ls.to(device), masks.to(device)
            logits = model(ws, cs, mask=masks)
            preds = torch.argmax(logits, dim=-1)  # b x seq_len

            for i in range(ws.size(0)): 
                sent_len = masks[i].sum().item()
                gold_labels = [id2label[id.item()] for id in ls[i][:sent_len]]
                pred_labels = [id2label[id.item()] for id in preds[i][:sent_len]]
                
                y_true.append(gold_labels)
                y_pred.append(pred_labels)

    print(classification_report(y_true, y_pred, digits=2))

In [14]:
id2label = {v:k for k,v in label_vocab.items()}

evaluate_seqeval(model, test_loader, id2label, device)

              precision    recall  f1-score   support

         DNA       0.80      0.82      0.81      1622
         RNA       0.84      0.68      0.75       322
   cell_line       0.77      0.84      0.80       808
   cell_type       0.88      0.88      0.88      4140
     protein       0.85      0.83      0.84     10504

   micro avg       0.85      0.84      0.85     17396
   macro avg       0.83      0.81      0.82     17396
weighted avg       0.85      0.84      0.85     17396

